# 2장 생존 자료와 Kaplan-Meier (R과 통계분석)

survival 패키지의 폐암 자료(lung)로 Kaplan-Meier 곡선을 그리고 군의 생존을 비교합니다. 보건통계학 기초의 파이썬 실습 7과 같은 자료입니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요.


## a. 생존 자료를 R에 알려 주기

### 셀 1. 폐암 자료 불러오기

In [ ]:
library(survival)
head(lung, 3)
table(lung$status)

### 셀 2. Surv()로 생존 자료 만들기

In [ ]:
lung$months <- lung$time / 30.4375      # 일 → 개월 (365.25 / 12)
s <- Surv(lung$months, lung$status)
head(s)
table(s[, "status"])

### 셀 3. 중앙 추적기간 (역 Kaplan-Meier)

In [ ]:
survfit(Surv(months, status == 1) ~ 1, data = lung)   # 중도절단을 사건으로 본 역 KM

## b. Kaplan-Meier 곡선과 중앙생존기간

### 셀 4. Kaplan-Meier 추정과 시점별 생존율

In [ ]:
km <- survfit(Surv(months, status) ~ 1, data = lung)
km
summary(km, times = c(6, 12, 24))

### 셀 5. 신뢰구간을 log-log 방식으로

In [ ]:
km_ll <- survfit(Surv(months, status) ~ 1, data = lung, conf.type = "log-log")
km_ll
summary(km_ll, times = c(6, 12, 24))

### 셀 6. 기본 그림으로 곡선 그리기

In [ ]:
plot(km_ll, xlab = "Months", ylab = "Survival probability",
     mark.time = TRUE, xaxt = "n")
axis(1, at = seq(0, 36, by = 6))
abline(h = 0.5, lty = 3, col = "gray50")

## c. 군별 곡선과 요약표

### 셀 7. 성별 Kaplan-Meier

In [ ]:
lung$sex_f <- factor(lung$sex, levels = 1:2, labels = c("Male", "Female"))
km_sex <- survfit(Surv(months, status) ~ sex_f, data = lung, conf.type = "log-log")
print(km_sex, rmean = 24)

### 셀 8. 논문에 옮길 성별 요약표

In [ ]:
st  <- summary(km_sex)$table           # 군별 n, 사건 수, 중앙값과 구간
s12 <- summary(km_sex, times = 12)     # 12개월 생존율과 구간
data.frame(group = levels(lung$sex_f),
           n = st[, "n.max"], deaths = st[, "events"],
           `median (95% CI)` = sprintf("%.1f (%.1f-%.1f)",
                                       st[, "median"], st[, "0.95LCL"], st[, "0.95UCL"]),
           `1-yr % (95% CI)` = sprintf("%.1f (%.1f-%.1f)",
                                       100 * s12$surv, 100 * s12$lower, 100 * s12$upper),
           check.names = FALSE, row.names = NULL)

### 셀 9. survminer로 곡선과 위험집합 표

In [ ]:
install.packages("survminer")
library(survminer)
ggsurvplot(km_sex, data = lung, conf.int = TRUE, pval = TRUE,
           risk.table = TRUE, break.time.by = 6,
           legend.title = "Sex", legend.labs = c("Male", "Female"),
           xlab = "Months", palette = c("#1f5f8b", "#c8553d"))

## d. 로그순위 검정과 그 밖의 비교

### 셀 10. 로그순위 검정

In [ ]:
lr <- survdiff(Surv(months, status) ~ sex_f, data = lung)
lr
lr$pvalue

### 셀 11. 초기 차이에 무게를 둔 검정 (rho = 1)

In [ ]:
survdiff(Surv(months, status) ~ sex_f, data = lung, rho = 1)

### 셀 12. 세 군 비교와 두 군씩 비교

In [ ]:
lung$ecog <- factor(ifelse(lung$ph.ecog >= 2, "2+", lung$ph.ecog))
table(lung$ecog, useNA = "ifany")
survdiff(Surv(months, status) ~ ecog, data = lung)
pairwise_survdiff(Surv(months, status) ~ ecog, data = lung,
                  p.adjust.method = "bonferroni")

### 셀 13. 24개월 제한평균생존시간(RMST)의 차이

In [ ]:
rm <- summary(km_sex, rmean = 24)$table[, c("rmean", "se(rmean)")]
rm
d  <- rm["sex_f=Female", "rmean"] - rm["sex_f=Male", "rmean"]
se <- sqrt(sum(rm[, "se(rmean)"]^2))
round(c(diff = d, lower = d - 1.96 * se, upper = d + 1.96 * se), 2)

## e. 과제 정답

### 셀 14. 과제 1 정답: ECOG 군별 요약표

In [ ]:
km_ecog <- survfit(Surv(months, status) ~ ecog, data = lung, conf.type = "log-log")
st  <- summary(km_ecog)$table
s12 <- summary(km_ecog, times = 12)
data.frame(group = rownames(st), n = st[, "n.max"], deaths = st[, "events"],
           `median (95% CI)` = sprintf("%.1f (%.1f-%.1f)",
                                       st[, "median"], st[, "0.95LCL"], st[, "0.95UCL"]),
           `1-yr % (95% CI)` = sprintf("%.1f (%.1f-%.1f)",
                                       100 * s12$surv, 100 * s12$lower, 100 * s12$upper),
           check.names = FALSE, row.names = NULL)

### 셀 15. 과제 2 정답: ECOG로 층화한 로그순위 검정

In [ ]:
survdiff(Surv(months, status) ~ sex_f + strata(ecog), data = lung)
survdiff(Surv(months, status) ~ sex_f, data = lung, subset = !is.na(ecog))

### 셀 16. 과제 3 정답: 사건 변수를 잘못 만들면

In [ ]:
lung$dead_wrong <- as.integer(lung$status > 0)    # 실수: 모두 1이 된다
table(lung$dead_wrong)
survfit(Surv(months, dead_wrong) ~ 1, data = lung)